# Augur — Cell type perturbation ranking

Quantifies which immune cell types are most transcriptionally perturbed
across injury conditions and timepoints using
[Augur](https://github.com/neurorestore/Augur) (via `pertpy`).

Augur trains a random forest classifier to distinguish two conditions from
gene expression alone. The AUC per cell type reflects how strongly that cell
type is transcriptionally perturbed — AUC near 0.5 = not perturbed,
AUC near 1.0 = highly perturbed.

**Analysis:** compares `inj_2` vs `inj_1` at each injury phase
(acute / subacute / late) to identify which immune cell types are
differentially perturbed between the two injury models.

**Input:** annotated immune cell `.h5ad` with:
- `adata.obs['broad_immune']` or `adata.obs['immune_subset']` — cell type labels
- `adata.obs['inj_type']` — injury type (`inj_2` / `inj_1`)
- `adata.obs['day']` — timepoint (`d1`, ..., `d14`)
- `adata.layers['transformed']` — log-normalized counts

## 1 · Imports & settings

In [ ]:
import scanpy as sc
import pertpy as pt
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)


## 2 · Load data

In [ ]:
adata_immune = sc.read_h5ad("/path/to/your/annotated_immune_cells.h5ad")

In [ ]:
adata_immune

## 3 · Prepare metadata

Assign phase labels and collapse uninjured variants.

In [ ]:
phase_map = {
    "uninjured": "uninjured",
    "d1": "acute",    "d2": "acute",    "d3": "acute",
    "d4": "subacute", "d5": "subacute", "d7": "subacute",
    "d9": "late",     "d14": "late",
}

adata_immune.obs["phase"] = adata_immune.obs["day"].map(phase_map).astype(str)

print(adata_immune.obs["phase"].value_counts())

In [ ]:
# Convert to plain string first (removes categorical dtype entirely)
adata_immune.obs['inj_type'] = adata_immune.obs['inj_type'].astype(str)
adata_immune.obs['phase'] = adata_immune.obs['phase'].astype(str)

# Now collapse all uninjured_* into 'uninjured'
mask_uninjured = adata_immune.obs['inj_type'].str.startswith('uninjured')
adata_immune.obs.loc[mask_uninjured, 'inj_type'] = 'uninjured'
adata_immune.obs.loc[mask_uninjured, 'phase'] = 'uninjured'

# Sanity check
print(adata_immune.obs.groupby(['inj_type', 'phase']).size())

## 4 · QC — cell counts per condition

In [ ]:
import pandas as pd

# Fix obs names first
adata_immune.obs_names_make_unique()

# Now check counts
for inj in ['inj_1', 'inj_2']:
    print(f"\n{'='*50}")
    print(f"  {inj.upper()}")
    print(f"{'='*50}")
    subset = adata_immune[adata_immune.obs['inj_type'].isin([inj, 'uninjured'])].copy()
    ct = pd.crosstab(subset.obs['broad_immune'], subset.obs['phase'])
    print(ct)
    
    print("\n⚠️  Cell types with < 10 cells in at least one phase:")
    for col in ct.columns:
        low = ct[ct[col] < 18].index.tolist()
        if low:
            print(f"  {col}: {low}")

## 5 · Run Augur — inj_2 vs inj_1 per phase

# Run AUGUR and compared injur model per phase

In [ ]:
AUGUR_DIR = "/path/to/your/augur/output/"
os.makedirs(AUGUR_DIR, exist_ok=True)

# Phases to compare inj_2 vs inj_1 at each phase
phases = ["acute", "subacute", "late"]

ag      = pt.tl.Augur("random_forest_classifier")
results = {}

for phase in phases:
    # Subset to inj_2 + inj_1 at this phase
    subset = adata_immune[
        (adata_immune.obs['inj_type'].isin(["inj_2", "inj_1"])) &
        (adata_immune.obs['phase'] == phase)
    ].copy()

    if subset.n_obs < 18:
        print(f"⚠️  Skipping {phase} — only {subset.n_obs} cells")
        continue

    subset.obs_names_make_unique()
    subset.obs['cell_type'] = subset.obs['immune_subset'].astype(str)
    subset.obs['label']     = subset.obs['inj_type'].map({"inj_2": 0, "inj_1": 1})
    subset.X                = subset.layers['transformed']

    print(f"Running: inj_2 vs inj_1 | {phase} | {subset.n_obs} cells")
    print(f"  inj_2: {(subset.obs['inj_type']=='inj_2').sum():,} | "
          f"inj_1: {(subset.obs['inj_type']=='inj_1').sum():,}")

    try:
        loaded       = ag.load(subset)
        result, clf  = ag.predict(
            loaded,
            subsample_size = 10,
            n_threads      = 4,
            random_state   = 42,
            span = 0.5,
            min_cells = 18,
            
        )

        summary = result.uns['summary_metrics'].T[['mean_augur_score']].reset_index()
        summary.columns   = ['cell_type', 'mean_augur_score']
        summary['phase']  = phase
        summary['comparison'] = "inj2_vs_inj1"
        results[phase]    = summary

        fname = f"augur_inj2_vs_inj1_{phase}.csv"
        summary.to_csv(os.path.join(AUGUR_DIR, fname), index=False)
        print(f"✅ Done: inj_2 vs inj_1 | {phase}")
        print(summary.sort_values("mean_augur_score", ascending=False).to_string())

    except Exception as e:
        print(f"❌ Failed {phase}: {e}")

# Save all combined
if results:
    all_results = pd.concat(results.values(), ignore_index=True)
    all_results.to_csv(os.path.join(AUGUR_DIR, "augur_inj2_vs_inj1_all.csv"), index=False)
    print(f"\nAll results saved → {AUGUR_DIR}")

## 6 · Visualise results

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
x_min, x_max = 0.45, 0.7   # ← adjust x scale here

color_map_augur = {
    'DC_T_cells'                : "#87C254",
    'Macro_Lipid_handling'      : "#FFC80A",
    'Macro_Il33'                : "#FF9500",
    'Macro_Cholesterol_handling': "#CC2200",
    'Macro_ECM_secreting'       : "#97000A",
    "MG_Homeostatic"          : "#C5DCFB",
    "MG_Migrating"            : "#4AA2AA",
    "MG_Inflammatory"         : "#1A9EFF",
    "MG_Lipid_handling"       : "#17355A",
    "MG_Cholesterol_handling" : "#0045FF",
    
}

cell_order = list(color_map_augur.keys())

# ── Load results ──────────────────────────────────────────────────────────────
if not results:
    all_results = pd.read_csv(os.path.join(AUGUR_DIR, "augur_inj2_vs_inj1_all.csv"))
else:
    all_results = pd.concat(results.values(), ignore_index=True)

# ── Plot ──────────────────────────────────────────────────────────────────────
phases_found = ["acute", "subacute", "late"]
phases_found = [p for p in phases_found if p in all_results["phase"].unique()]

fig, axes = plt.subplots(
    1, len(phases_found),
    figsize=(5 * len(phases_found), 5),
    sharey=False
)
if len(phases_found) == 1:
    axes = [axes]

for ax, phase in zip(axes, phases_found):
    data = all_results[all_results["phase"] == phase].copy()

    # Apply fixed order — only keep cell types present in results
    order = [ct for ct in cell_order if ct in data["cell_type"].values]
    data  = data.set_index("cell_type").reindex(order).reset_index()
    colors = [color_map_augur[ct] for ct in data["cell_type"]]

    ax.barh(
        data["cell_type"],
        data["mean_augur_score"],
        color     = colors,
        edgecolor = "white",
        linewidth = 0.5,
    )
    ax.axvline(0.5, color="black", linewidth=1, linestyle="--", alpha=0.7)
    ax.set_xlim(x_min, x_max)
    ax.set_xlabel("Mean Augur score (AUC)", fontsize=10)
    ax.set_title(phase, fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)    
    ax.invert_yaxis()

axes[0].set_ylabel("Cell type", fontsize=10)
plt.suptitle("Augur scores — inj_2 vs inj_1", fontsize=14)
plt.tight_layout()
plt.savefig(
    os.path.join(AUGUR_DIR, "augur_inj2_vs_inj1_barplot.pdf"),
    bbox_inches="tight", dpi=300, format="pdf"
)
plt.show()